# Objectives

- Understand the fundamental interactions with overcollateralized borrow-lending protocols
- Combining DeFi elements to perform a short on a token
- Perform a profitable arbitrage between different Uniswap pools using a Flash Loan from Aave

We will interact with Mutuum Finance Web App to perform the borrowing and lending operations, and using Aave's Flash Loans to perform the arbitrage.

⚠️ **Important Disclaimers:**

For Educational Purposes Only: The products and protocols mentioned in this experiment (e.g., Mutuum Finance, Aave) are used strictly for demonstration and educational purposes. There is no recommendation or endorsement implied for any of these platforms.


#### Prerequisites 

- Metamask 
- Ensure a minimum of 0.1 `SepoliaETH` in your account


#### Important Contract Addresses:

- `mtWETH`: 0x03748f48D57259694daf5c703308FEaB5576A765
- `Sepolia USDT`: 0xaA8E23Fb1079EA71e0a56F48a2aA51851D8433D0
- `Sepolia USDC`: 0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8
- `PoolAddressesProvider`: 0x012bAC54348C0E635dCAc9D5FB99f06F24136C9A
- `Aave Pool`: 0x6Ae43d3271ff6888e7Fc43Fd7321a503ff738951

----
----

# Collateralized Borrowing on Mutuum Finance

- Mutuum Finance: https://app.mutuum.com/ 

### 1. Objective
The goal of this part is to understand the mechanics of overcollateralized lending in DeFi. We will interact with the Mutuum Finance testnet to supply ETH as collateral, borrow USDT against it, and manage our debt position through repayment and withdrawal.

### 2. Core Concepts & Pre-reading
Before diving into the steps, let's clarify a few fundamental concepts. 

- **Pre-approved Collateral:** You might wonder, why does the contract only accept a broad set of pre-approved tokens as collateral? This is a critical risk management design. Not all tokens have sufficient liquidity or reliable price feeds (oracles). If the platform accepted highly volatile or illiquid tokens, the protocol could accrue bad debt.
- **Yield & Debt Tokenization:** aTokens (e.g., aWETH): When you supply assets, you receive interest-bearing aTokens mapped 1:1 to your supplied asset. These tokens continuously compound interest directly in your wallet.
- **Debt Tokens (e.g., variableDebtUSDT):** When you borrow, you receive Debt tokens. These are non-transferable tokens that track your exact borrowed amount plus the accrued interest you owe to the protocol.
- **Interest Rate Spread:** The supply APY (Annual Percentage Yield) is mathematically designed to be lower than the borrow APY. The difference accounts for the protocol's reserve factor (treasury revenue) and the fact that not 100% of supplied funds are borrowed at any given time (Utilization Rate).


### 3. Step-by-Step Guide Interacting with Mutuum Finance

#### Phase 1: Supplying Collateral

To borrow funds you must first deposit an asset to back your loan.

1. **Connect Wallet:** Navigate to the Mutuum Finance web app at https://app.mutuum.com/ and connect your Web3 wallet.
2. **Explore the UI:** Observe the dashboard. You could click on the `Details` button under specific Tokens (e.g., ETH) to see more information about the asset. You will find the following parameters:
   - Max LTV 80%: maximum borrowing power of a specific collateral.
   - Liquidation Threshold 85%: if the value of your collateral falls below this threshold, your position becomes vulnerable to liquidation.
   - Liquidation Penalty 5%: if your position is liquidated, a portion of your collateral (5%) will be taken as a penalty.
3. **Supply ETH:** On the dashboard, we find the actions: `Supply`, `Borrow`, `Repay`, and `Withdraw`. Click on the `Supply` button for ETH and supply 0.01 ETH to the Mutuum Finance smart contract. 
   - We can browse the transaction details: We transfer 0.01 `WETH` to the Mutuum Finance contract, and in return, we receive `Pyraxe Interest Bearing WETH (mtWETH)` (0x03748f48D57259694daf5c703308FEaB5576A765) in our wallet. 
   - `mtWETH` is an interest-bearing token that represents our supplied collateral. It continuously accrues interest, and its value increases over time as we earn yield on our supplied ETH.


#### Phase 2: Borrowing Assets

Now that your account has collateral, your borrowing power is unlocked.

1. **Understand LTV (Loan-to-Value):** Mutuum Finance assigns specific LTV ratios to different assets. For ETH in this environment, the protocol allows you to borrow funds up to 80% of your collateral's USD value.
2. **Execute the Borrow:** Go back to the main dashboard. Choose `USDT` and borrow 2 USDT (0xaA8E23Fb1079EA71e0a56F48a2aA51851D8433D0).
   - Stability Factor: a position’s stability factor reflects the ratio between the adjusted collateral value and the outstanding debt. If the stability factor slips below a critical level, liquidation may occur. 
3. **Check Balances:** Check your wallet. You now have 2 actual USDT tokens that you can use freely (trade on an AMM, transfer, or re-supply). Simultaneously, the protocol has minted `variableDebtmtUSDT` (0xbae4Ca6F13D0571A83f7c3Bdc641557aEA4AB3a4) to your address to track your open loan and its continuously accruing interest.


#### Phase 3: Position Management (Repay & Withdraw)
You cannot withdraw all your underlying ETH collateral as long as you have an active debt position that relies on it. To unlock your ETH, you must close the loan.
1. **Repay the Loan:** Click on your borrowed USDT position to initiate a repayment. Note that to clear the entire debt, you would need slightly more than 2 USDT because of the accrued interest. For the sake of this lab's brevity, just execute a partial repayment of **1 USDT**.
   - The `Stability Factor` will improve as you reduce your outstanding debt.
   - Both `variableDebtmtUSDT` and your actual `USDT` balance will decrease by 1 USDT after this transaction.
2. **Withdraw Collateral:** With your debt significantly reduced, your `Stability Factor` improves, freeing up more of your collateral. Go to your supplied ETH and initiate a withdrawal of 0.001 ETH. You can also find withdrawal will reduce your `Stability Factor`.
3. **Observe Contract Logic:** Notice that under the hood, this transaction burns 0.001 `mtWETH` from your wallet and unlocks the equivalent native `WETH/ETH` back to you.


---
---



## Performing a short

Assume that we know the price of USDT in USTUSD will drop in this pool, we can first borrow 1 USDT from Mutuum Finance, then swap it for USTUSD in the Uniswap V3 pool, and finally repay the borrowed USDT to close the position. 

Borrow-lending platforms can be used to perform a short on any token that we can borrow. In this part, we will be shorting 1 USDT. Please follow the steps below to perform a short.


### Get current USDT price in USTUSD from Uniswap V3 Pool

In [ ]:
from web3 import Web3
import json
import os

infura_key = ''
wallet_public_address = Web3.to_checksum_address('')
wallet_private_key = ''

USDC_address = Web3.to_checksum_address('0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8')
USDT_address = Web3.to_checksum_address('0xaA8E23Fb1079EA71e0a56F48a2aA51851D8433D0') 
USTUSD_address = Web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')
web3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))
print("Connected to Sepolia Testnet:", web3)
abi_file_path = os.path.join('./abis.json')
try:
    with open(abi_file_path, 'r', encoding='utf-8') as f:
        abi_data = json.load(f)
    print("ABI loaded successfully.")
except Exception as e:
    print(f"Error loading ABI: {e}")

USDT_contract = web3.eth.contract(address=USDT_address, abi=abi_data['ERC20_ABI'])
USTUSD_contract = web3.eth.contract(address=USTUSD_address, abi=abi_data['ERC20_ABI'])

Connected to Sepolia Testnet: <web3.main.Web3 object at 0x0000023E26C36C20>
ABI loaded successfully.


In [17]:
factory_addr = '0x0227628f3F023bb0B980b67D528571c95c6DaC1c'
factory_contract = web3.eth.contract(factory_addr, abi=abi_data['UNISWAP_FACTORY_ABI'])
def get_pool_address(tokenA, tokenB, tier_fee, factory_contract):
    # Ensure tokens are in correct order (Uniswap V3 requires sorted token addresses)
    # tier_fee: 100 for 0.01%, 500 for 0.05%, 3000 for 0.3%, 10000 for 1%
    if tokenA > tokenB:
        tokenA, tokenB = tokenB, tokenA

    # Call the getPool function
    pool_address = factory_contract.functions.getPool(tokenA, tokenB, tier_fee).call()
    return pool_address

USDT_USTUSD_pool_address = get_pool_address(USTUSD_address, USDT_address, 500, factory_contract)
print("USDT-USTUSD Pool Address:", USDT_USTUSD_pool_address)

USDT_USTUSD_pool = web3.eth.contract(address=USDT_USTUSD_pool_address, abi=abi_data['UNISWAP_V3_POOL_ABI'])

USDT_price_in_USTUSD = USDT_USTUSD_pool.functions.slot0().call()[0]**2 / 2**192 / (10**12)
print(f"1 USDT = {USDT_price_in_USTUSD:.6f} USTUSD")

USDT-USTUSD Pool Address: 0x49e75DCCCf6Bb59531dC52Ea85579Bc460A59ccF
1 USDT = 4.162606 USTUSD


In [16]:
original_price = USDT_price_in_USTUSD

### Swap USDT for USTUSD in Uniswap V3 Pool (Similar to Lab 4)

In [ ]:
# approve the Universal Router to spend USDT on behalf of the wallet
universal_router_address = web3.to_checksum_address('0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD')
universal_router_contract = web3.eth.contract(address=universal_router_address, abi=abi_data['UNIVERSAL_ROUTER_ABI'])
permit2_address = web3.to_checksum_address('0x000000000022D473030F116dDEE9F6B43aC78BA3')
permit2_contract = web3.eth.contract(address=permit2_address, abi=abi_data['PERMIT2_ABI'])

def send_tx(tx, private_key):
    signed_txn = web3.eth.account.sign_transaction(tx, private_key)
    raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
    tx_hash = web3.eth.send_raw_transaction(raw_transaction)
    receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
    print(f"Transaction sent: {tx_hash.hex()} ", end=" ; ")
    receipt_status = "success" if receipt.status == 1 else "failure"
    print(f"Transaction Status: {receipt_status}") 
    return tx_hash, receipt

def approve_token_spending(web3, contract, wallet_public_address, *approve_args):
    tx = contract.functions.approve(*approve_args).build_transaction({
        "from": wallet_public_address,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
    })
    tx_hash, receipt = send_tx(tx, wallet_private_key)


approve_token_spending(web3, USDT_contract, wallet_public_address, permit2_address, 2**256 - 1)
approve_token_spending(web3, permit2_contract, wallet_public_address, USDT_address, universal_router_address, 2**160 - 1, 2**48 - 1)
approve_token_spending(web3, USTUSD_contract, wallet_public_address, permit2_address, 2**256 - 1)
approve_token_spending(web3, permit2_contract, wallet_public_address, USTUSD_address, universal_router_address, 2**160 - 1, 2**48 - 1)

Transaction sent: 2417666f48e44a23f758bf3a11f28b509f97020a2d63947eb18e36232d09eccc
Transaction Status: success
Transaction sent: bc6d6b2741b64e9691a94c12d3be2e2fa503a9395fd33632d7c9d28363cf55a2
Transaction Status: success
Transaction sent: b103d6c542e04945956f1986f9ad63d674551eb353b42ca3eb1a2780b4b3dedc
Transaction Status: success
Transaction sent: 37b458aa9f17554c1d551b68c2f4e7bd36bcb0a2fc5d20a9e324d130547616d3
Transaction Status: success


In [15]:
# build the transaction data for the swap using the Universal Router's encoding functions
from uniswap_universal_router_decoder import FunctionRecipient, RouterCodec
codec = RouterCodec()

encoded_data = codec.encode.chain().v3_swap_exact_in(
        FunctionRecipient.SENDER,  # reciver
        1*10**6,  # amount in, (1 USDT with 6 decimals)
        0,
        [   
            USDT_address,
            500,
            USTUSD_address
        ],
    ).build(2**256 - 1)

tx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": web3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }

tx_hash, receipt = send_tx(tx_params, wallet_private_key)
if receipt.status == 1:
    print(f"Swapped 1 USDT for USTUSD successfully! Check your wallet balance.")
else:
    print("Swap failed.")


Transaction sent: 8794daac4e0379082e0a0b1789adfed3598603274228a2cef2647c9b104ad47c
Transaction Status: success
Swapped 1 USDT for USTUSD successfully! Check your wallet balance.


### Waiting for the price dip of USDT (GTA will simulate this by posting a large swap on Uniswap)

### Transfer: USTUSD -> USDT

In [18]:
USTUSD_in_amount = int(original_price * 10**18)

encoded_data = codec.encode.chain().v3_swap_exact_in(
        FunctionRecipient.SENDER,  # reciver
        USTUSD_in_amount, 
        0,
        [   
            USTUSD_address,
            500,
            USDT_address
        ],
    ).build(2**256 - 1)

trx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": web3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }

tx_hash, receipt = send_tx(trx_params, wallet_private_key)


Transaction sent: 55b18c0a9b406f58049e5a812d8297007d500abfbb405d7172397fc53eb7493d
Transaction Status: success


Check the balance of USDT in your wallet. You should have more than 1 USDT now. Repay the 1 USDT back to Mutuum Finance to close the position. You can also choose to repay a smaller amount (e.g., 0.5 USDT) and keep the position open to earn yield on the remaining collateral.

---
---


# Flash Loans

## Background

[Flash Loans](https://docs.aave.com/developers/guides/flash-loans) are special transactions that allow the borrowing of an asset, as long as the borrowed amount (and a fee) is returned before the end of the transaction (also called One Block Borrows). These transactions do not require a user to supply collateral prior to engaging in the transaction. There is no real world analogy to Flash Loans, so it requires some basic understanding of how state is managed within blocks in blockchains.

## Execution Flow

Flash Loans typically operate similar to the following:

- Your contract calls the public Flash Loan function (eg. `flashLoan`) on the protocol's contract
- The protocol's contract sends you the requested funds and then calls a designated callback function defined in your contract (eg. `executeOperation`)
- The protocol's contract ensures that you have enough funds to repay the borrowed amount (plus a fee) and reclaims those funds

The `executeOperation` function in your contract can perform any arbitrary logic using the funds. Once it finishes, execution returns back to the calling protocol contract where it can transfer the borrowed funds back to itself and ensure the entire amount is repaid. If you no longer have enough funds to repay the borrowed amount by the last step, the entire transaction will fail. As a result, the protocol can let anyone borrow any amount with no risk of losing assets.

## Flashloan Contract

Your contract that receives the flash loaned amounts must conform to the [IFlashLoanSimpleReceiver.sol](https://github.com/aave-dao/aave-v3-origin/blob/main/src/contracts/misc/flashloan/interfaces/IFlashLoanSimpleReceiver.sol) or IFlashLoanReceiver.sol interface by implementing the relevant `executeOperation` function.

When using a standard ethereum account, we need to send a transaction to the appropriate Pool to invoke the flashLoan() or [flashLoanSimple()](https://docs.aave.com/developers/core-contracts/pool#flashloansimple) function.


Below is a typical flash loan smart contract, in which we need to implement our arbitrage function in `executeOperation`.

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.4;

import "@openzeppelin/contracts/utils/math/SafeMath.sol";
import "@aave/core-v3/contracts/flashloan/base/FlashLoanSimpleReceiverBase.sol";
import "@openzeppelin/contracts/token/ERC20/IERC20.sol";

contract SimpleFlashLoan is FlashLoanSimpleReceiverBase {
    using SafeMath for uint256;
    event Log(address asset, uint256 val);

    constructor(IPoolAddressesProvider provider)
        FlashLoanSimpleReceiverBase(provider)
    {}

    function createFlashLoan(address asset, uint256 amount) external {
        address receiver = address(this);
        bytes memory params = "";
        uint16 referralCode = 0;

        emit Log(asset, IERC20(asset).balanceOf(address(this)));

        POOL.flashLoanSimple(receiver, asset, amount, params, referralCode);
    }

    function executeOperation(
        address asset,
        uint256 amount,
        uint256 premium,
        address initiator,
        bytes calldata params
    ) external returns (bool) {
        // run arbitrage or liquidations here
        // abi.decode(params) to decode params

        emit Log(asset, IERC20(asset).balanceOf(address(this)));

        uint256 amountOwing = amount.add(premium);
        IERC20(asset).approve(address(POOL), amountOwing);

        return true;
    }
}
```

**Illustration of Flash Loan Execution Flow:**

#### Initiating the Loan: `createFlashLoan`
This is the entry point where you tell Aave what you want to borrow.
* **Asset:** The token address (e.g., USDC or DAI).
* **Amount:** How much you want to borrow.
* **POOL.flashLoanSimple:** This line calls the Aave pool. Aave will instantly send the funds to your contract and then immediately call your `executeOperation` function.

#### The Logic Loop: `executeOperation`
This is the most critical part of the contract. It follows a specific "callback" pattern:

1.  **Receive Funds:** When this function starts, your contract already has the borrowed tokens.
2.  **Custom Logic:** Where the comment says `// run arbitrage or liquidations`, you would add your own code to trade those funds on Uniswap or liquidate a loan to make a profit.
3.  **Repayment:** You calculate the total debt: $Total = Amount + Premium$ (where "premium" is Aave's fee).
4.  **Approval:** You call `approve` to allow the Aave pool to pull the funds back from your contract.
5.  **Final Check:** If the contract doesn't have enough funds to pay back the loan at the end of this function, the entire transaction **reverts** (it's as if the loan never happened).


### Lab Exercise: Arbitrage between Uniswap V3 Pools using Aave Flash Loans
In this lab, we will conduct a profitable arbitrage across Uniswap `USDC/USTUSD ` pools. We've established two `USDC/USTUSD ` pools with fees of 0.05% and 0.3% on the Sepolia Testnet and deliberately created an arbitrage opportunity by altering the swap rates to differ between the two pools.

We will leverage a Flash Loan of `USDC` from Aave in order to perform the arbitrage between two pools without needing to have any of either token to start with.

The arbitrage is as follows (assume `USDC/USTUSD ` is cheaper on Uniswap):

- Deploy a Flashloan smart contract
- Flash Loan 1000000(uint 256) amount of `USDC`(same to 1 USDC) from Aave
- Swap all our `USDC` for `USTUSD ` on the first pool 
- Swap all our `USTUSD ` for `USDC` on the second pool
- Repay the borrowed 1 `USDC` + a small fee to the Aave contract 
- Get the profit from the deployed smart contract to your account

If we cannot afford to repay what we borrowed in the second to last step, the whole transaction will revert. If the transaction does not revert, then we must have made a profit! Note that our profit is denominated in `USDC`. Optionally, we could include a final step to swap our profit into some other token, such as a stablecoin.

Importantly, note that:
- We want to use the [`flashLoanSimple`](https://docs.aave.com/developers/core-contracts/pool#flashloansimple) function defined in the Aave Pool contract
- The `flashLoanSimple` will call the callback function in our contract named `executeOperation`
- We **do not** need to transfer the borrowed funds back; the Aave contract will automatically take back the borrowed funds (assuming we have enough funds)

# Interacting with Uniswap using Web3 API

We continue this section from week5. We understand how to build transactions and approve the right smart contracts to use our funds to perform the arbitrage. Then, we execute an atomic swap to generate a profit. This time, we will use Flashloan .

## Which pools to arbitrage?

We first see the list of pools that exchange USDC and USTUSD . Then, we work out the best way to do arbitrage so that we make a profit at the end. The following cell should print the address of the two pools we are interested in.

In [ ]:
usdc_USTUSD_pool_address_1 = get_pool_address(USTUSD_address, USDC_address, 100, factory_contract)
usdc_USTUSD_pool_address_2 = get_pool_address(USTUSD_address, USDC_address, 10000, factory_contract)

print("Uniswap V3 Pool address with fee 0.01% :",usdc_USTUSD_pool_address_1)
print("Uniswap V3 Pool address with fee 0.1% :",usdc_USTUSD_pool_address_2)

Now, we print the price of USDC on both these pools.

In [ ]:
univ3_pool_1 = web3.eth.contract(address=usdc_USTUSD_pool_address_1, abi=abi_data['UNISWAP_V3_POOL_ABI'])

current_price_1 = univ3_pool_1.functions.slot0().call()[0]**2 / 2**192 / (10**12)

print(f"1 USDC = {current_price_1} USTUSD in the 0.01% fee pool")


In [ ]:
univ3_pool_2 = web3.eth.contract(address=usdc_USTUSD_pool_address_2, abi=abi_data['UNISWAP_V3_POOL_ABI'])

current_price_2 = univ3_pool_2.functions.slot0().call()[0]**2 / 2**192 / (10**12)

print(f"1 USDC = {current_price_2} USTUSD in the 0.1% fee pool")

As we see here, `USDC` is worth more `USTUSD ` on pool with fee 0.01%, than it is on 0.1% right now.

To take advantage of this, we want to *borrow* `USDC`, *sell* `USDC` in exchange for `USTUSD ` on one pool, *buy* `USDC` using the `USTUSD ` on another pool, return the original amount of `USDC` we borrowed, and then our profit is the remaining `USDC`.

Furthermore, we want to use a Flash Loan for that first step, so we don't have to provide any of our own capital. To use a Flash Loan, we must call `flashLoanSimple` on the Aave contract, and that call must come from another contract since we must have a callback function named `executeOperation` for it to call, so our arbitrage logic must take place entirely within a smart contract rather than using an EOA and multiple transactions using `web3.py`.

## Contract

The contract below implements all the logic described up to this point. Please read through the smart contract, fill in the pool fee, and follow the instructions for deployment.

```javascript
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.4;

import "@openzeppelin/contracts/utils/math/SafeMath.sol";
import "@aave/core-v3/contracts/flashloan/base/FlashLoanSimpleReceiverBase.sol";
import "@openzeppelin/contracts/token/ERC20/IERC20.sol";
import "@uniswap/v3-periphery/contracts/interfaces/ISwapRouter.sol";
import "@uniswap/lib/contracts/libraries/TransferHelper.sol";
import "@uniswap/swap-router-contracts/contracts/interfaces/IV3SwapRouter.sol";

contract FlashloanArbitrage is FlashLoanSimpleReceiverBase {
    using SafeMath for uint256;

    uint256 constant MAX_UINT = 2**256 - 1;

    address private constant SWAP_ROUTER =
        0x3bFA4769FB09eefC5a80d6E87c3B9C650f7Ae48E;
    address private constant USTUSD  = 0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c;
    address public constant USDC = 0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8;
    address public owner;

    IV3SwapRouter public immutable swapRouter = IV3SwapRouter(SWAP_ROUTER);

    event ArbitrageResult(uint256 amountOut);
    event BalanceTransferred(address recipient, uint256 amount);

    constructor(IPoolAddressesProvider provider)
        FlashLoanSimpleReceiverBase(provider)
    {
        owner = msg.sender;
    }

    function createFlashLoan(address asset, uint256 amount) external {
        address receiver = address(this);
        bytes memory params = "";
        uint16 referralCode = 0;

        POOL.flashLoanSimple(receiver, asset, amount, params, referralCode);
    }

    function executeOperation(
        address asset,
        uint256 amount,
        uint256 premium,
        address initiator,
        bytes calldata params
    ) external returns (bool) {
        // Assume `doArbitrage` is adequately modified not to emit unnecessary logs.
        this.doArbitrage(1000000);

        uint256 amountOwing = amount.add(premium);
        IERC20(asset).approve(address(POOL), amountOwing);

        return true;
    }

    function doArbitrage(uint256 amountIn)
        external
        returns (uint256 amountOut)
    {
        IERC20(USDC).approve(address(swapRouter), MAX_UINT);
        IV3SwapRouter.ExactInputSingleParams memory params = IV3SwapRouter
            .ExactInputSingleParams({
                tokenIn: USDC,
                tokenOut: USTUSD ,
                fee: 100, //100 or 10000 in the first swap
                recipient: address(this),
                amountIn: amountIn,
                amountOutMinimum: 0,
                sqrtPriceLimitX96: 0
            });

        uint256 amountOut1 = swapRouter.exactInputSingle(params);

        TransferHelper.safeApprove(USTUSD , address(swapRouter), MAX_UINT);

        IV3SwapRouter.ExactInputSingleParams memory params2 = IV3SwapRouter
            .ExactInputSingleParams({
                tokenIn: USTUSD ,
                tokenOut: USDC,
                fee: 10000, //100 or 10000 in the second swap
                recipient: address(this),
                amountIn: amountOut1,
                amountOutMinimum: 0,
                sqrtPriceLimitX96: 0
            });

        amountOut = swapRouter.exactInputSingle(params2);

        emit ArbitrageResult(amountOut);
        return amountOut;
    }

    // Modifier to restrict access to the owner only
    modifier onlyOwner() {
        require(msg.sender == owner, "Caller is not the owner");
        _;
    }

    function transferUSDCBalance(address recipient) external onlyOwner {
        uint256 balance = IERC20(USDC).balanceOf(address(this));
        require(balance > 0, "No USDC balance to transfer");

        TransferHelper.safeTransfer(USDC, recipient, balance);

        emit BalanceTransferred(recipient, balance);
    }
}


```

## Instructions for deployment:
- Create a new project in Remix
- Fill in pool fee and compile the code
- Click on the deploy tab and select an Injected provider - Metamask on Sepolia
- Click on deploy(Aave pool provider: 0x012bAC54348C0E635dCAc9D5FB99f06F24136C9A) - observe the transaction on Etherscan and fetch the contract address
- Verify the contract on Etherscan - Follow the steps from the assignments
- Once verified, click on ``Contract``->``Write Contract`` and call the ``createFlashLoan`` function with ``amount`` as 1 USDC (Type: 1000000) with asset address `0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8`
- Check the transaction on Etherscan and observe the USDC and USTUSD  transferred in the transaction
- After the ``createFlashloan`` function call succeeded, click on ``transferUSDCBalance`` function with your account to gain profit
- Use the "Import Tokens" link at the bottom of your MetaMask. 
- Enter the USDC address(0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8) and observe the profit you gained